# Elemental maps of h-BN
## Elemental Quantification of a Linescan
This notebook is an example of how to get elemental maps from CL EELS data.

It starts with the getting a prediction for the background for two edges of two corresponding elements. 

The edges in this case are the boron K-edge and the nitrogen K-edge. these have a edge onset energy of 188eV and 402eV respectively.

It starts by importing the packages for the Core-loss fitter and the elemental quantification.

In [ ]:
import sys
sys.path.append(r'..\Code')
from CLfitter import *
from ElementalQuantification import *

path =r'..\Data\hBN_linescan_data.dm3'
data_handler = DataHandler()
data_handler.read_dm3_linescan(path)
data_handler.align_data_cross_correlate()

To start the code, the data first needs to be loaded. In this case it is a linescan, with a dataformat of DM3. 
Therefore, the DataHandler.read_dm3_linescan is used for reading the data.

after this the data is aligned using a cross correlation.
Also, the two regions are plotted, from this we can define the regions for the fitting were going to do.

In [ ]:

data_handler.plot_spectra(range(5), (0, 10000)) #Use this to search for suitable energy windows

# data_handler.plot_spectra(range(5), ())

data_handler.plot_intensity_histogram(bins_nr = 20) #Use this to find a good number of clusters. 
                                                    #For this dataset, the range is quite large, it then better to take a larger number, 
                                                    # correspodning to clusters with roughly equal width

We see in the plots above that the edges do not start exactly at 188 eV and 402 eV. This is due to shifts and spectral broadening.

for the code to work, we need two connected energy regions, the first on which the background is prediced, and the second on which the background is extrapolated. these regions are defined by a start, onset and stop energy.

We take respectively: 

We also plot the histogram, from this we can get a rough number of clusters needed. For this we take 6.

In [ ]:
regions = [ #regions (E_start, E_onset, E_stop), its better to take E_onset a bit (1eV) before 
    (XX, XX, XX),
    (XX, XX, XX)
]

signal = data_handler.signal

class EELSBackgroundNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear( 2, 12),
            nn.SiLU(),
            # nn.Dropout(0.2), 
            nn.Linear(12, 10),
            nn.SiLU(),
            # nn.Dropout(0.2), 
            nn.Linear(10,1),
        )
    def forward(self, x):
        return self.model(x)
    
for i in regions:
    range_mask = (data_handler.energy_axis > i[0]) & (data_handler.energy_axis < i[-1])  # Range for clustering shape [n_E_1]

    energy_range = data_handler.energy_axis[range_mask] # shape [n_E_1]
    signal_range = signal.copy()[range_mask,:]  # shape [n_E_1, n_y*n_x]

    pre_edge_mask = (energy_range > i[0]) & (energy_range < i[1])  # Range for pre-edge shape [n_E_2]

    clusterer = ClusterAnalyzer(signal_range)
    clusterer.cluster_data(n_clusters = XXX, 
                           pre_edge_mask=pre_edge_mask,)
    clusterer.cholesky_decomp()
    
    X_builder = X_Builder(energy_range)
    X_builder.prepare_X_mc_data(clusterer.cluster_centers, 
                                i[1])
    X_builder.prepare_X_eval_data(clusterer.total_integrated_intensity)
    
    background_trainer = BackgroundTrainer(
        signal=signal_range,
        pre_edge_mask=pre_edge_mask,
        X_mc=X_builder.X_mc,
        X_eval=X_builder.X_eval,
        clustered_spectra_mean=clusterer.clusters_mean,
        triangular_matices=clusterer.triangular_matices,
        covariance_matrices=clusterer.clusters_covariance,
        cluster_labels=clusterer.clusters,
    )

    background_trainer.train_MC_replica_consecutive(
        n_mc_replicas=20, 
        epochs=20000, 
        edge_onset=i[1],
        replica_version='triangular',
        progress = False, # These can be set to true to check how the method is doing. 
        logging = False, # This can useful for when the model architecture needs to be checked
        model = EELSBackgroundNN(),
    )    


    predictions = background_trainer.background

    prediction_saver = PredictionSaver(
        signal=signal_range,
        energy_axis=energy_range,
        spatial_axis_x=data_handler.spatial_axis_x,
        spatial_axis_y=data_handler.spatial_axis_y,
        predictions=predictions)
    

    path_to_save = path.replace('.dm3', f'_background_{i[0]}-{i[1]}-{i[2]}.npz')
    prediction_saver.save_predictions(path_to_save)

The code saves the predictions in an .npz file. Do note, it rewrites files of the same name.

Now we can take a look at the predictions, and see if they are any good.

In [ ]:
class PredictionChecker:
    def __init__(self, path):
        file = np.load(path)

        self.predictions = file['predictions']
        self.signal = file['signal']
        self.spatial_x = file['spatial_axis_x']
        self.spatial_y = file['spatial_axis_y']
        self.energy_axis = file['energy_axis']

        self.signal_mean = np.mean(self.signal[:,:,None] - self.predictions.T, axis = 2)
        self.signal_stdev = np.std(self.signal[:,:,None] - self.predictions.T, axis = 2)
    

    def plot_prediction(self, spectrum_idx):
        plt.figure(figsize = (10,6), dpi = 300)
        plt.plot(self.energy_axis, self.signal_mean[:,spectrum_idx])
        plt.fill_between(self.energy_axis,  
                         self.signal_mean[:,spectrum_idx]-self.signal_stdev[:,spectrum_idx],
                         self.signal_mean[:,spectrum_idx]+self.signal_stdev[:,spectrum_idx],
                         alpha = 0.4)



In [ ]:
path1 = path.replace('.dm3', f'_background_{regions[0][0]}-{regions[0][1]}-{regions[0][2]}.npz')
path2 = path.replace('.dm3', f'_background_{regions[1][0]}-{regions[1][1]}-{regions[1][2]}.npz')
PC = PredictionChecker(path2)
PC.plot_prediction(40)

With the predictions, we can get the elemental quantification. For this, we need an energy window that includes only the edge, but excludes the ELNES.

For this we take (XXX, XXX) and (XXX, XXX)

In [ ]:
GOS_path = r'../Data\Dirac_GOS.gosh'
# path =r'..\Data\008_core_loss_line_020evpx_1mm.dm3'
# regions = [ #regions (E_start, E_onset, E_stop)

# ]

path1 = path.replace('.dm3', f'_background_{regions[0][0]}-{regions[0][1]}-{regions[0][2]}.npz')

pipeline1 = CrossSectionFit(
    GOS_path=GOS_path,
    edge_path=path1,
    element='B',
    edge_labels=('K1'),
    beta=10e-3,
    E0=300e3,
    E_min=,
    E_max=,
    plot=False
)
scale_factor_mean_1, scale_factor_stdev_1 = pipeline1.run()

path2 = path.replace('.dm3', f'_background_{regions[1][0]}-{regions[1][1]}-{regions[1][2]}.npz')

pipeline2 = CrossSectionFit(
    GOS_path=GOS_path,
    edge_path=path2,
    element='N',
    edge_labels=('K1'),
    beta=10e-3,
    E0=300e3,
    E_min=,
    E_max=,
    plot=False
)

scale_factor_mean_2, scale_factor_stdev_2 = pipeline2.run()

In [ ]:
# pipeline2.plot_spectrum_with_fit(77, scale_factor_mean_2)

now we have the scale factors, and their associated uncertainty w.r.t. the Backgorund prediction. Now using some math we can get the spatially resolved elemental ratio between boron and nitrogen.

In [ ]:
# uncertainties for the cross sections.
cross_section_uncertainty_B = 0.1  # 10% uncertainty in B PCS
cross_section_uncertainty_N = 0.1 # 10% uncertainty in N PCS

# Compute ratio
ratio = scale_factor_mean_1 / scale_factor_mean_2

# Compute relative errors (statistical)
rel_err_B_stat = scale_factor_stdev_1 / scale_factor_mean_1
rel_err_N_stat = scale_factor_stdev_2 / scale_factor_mean_2

# Combine with systematic uncertainties
rel_err_total_B = np.sqrt(rel_err_B_stat**2 + cross_section_uncertainty_B**2)
rel_err_total_N = np.sqrt(rel_err_N_stat**2 + cross_section_uncertainty_N**2)

# Propagate combined uncertainty
ratio_error = ratio * np.sqrt(rel_err_total_B**2 + rel_err_total_N**2)

spatial_axis = np.load(path2)['spatial_axis_y']*1000

plt.figure(figsize=(10,6), dpi = 300)
plt.plot(spatial_axis, ratio, c='blue')
plt.fill_between(spatial_axis, ratio-ratio_error, ratio+ratio_error, color= 'blue', alpha = 0.3)
plt.xlabel('Spatial Position [nm]')
plt.ylabel('Ratio B/N')

In [ ]:
import matplotlib as mpl
GOS_path = r'../Data\Dirac_GOS.gosh'
path =r'..\Data\008_core_loss_line_020evpx_1mm.dm3'
regions = [ #regions (E_start, E_onset, E_stop)

]

mpl.rcParams["mathtext.fontset"] = "cm"   # Computer Modern math
mpl.rcParams["font.family"] = "serif"     # Match axes labels with math
plt.rcParams['font.size'] = 16 

path1 = path.replace('.dm3', f'_background_{regions[0][0]}-{regions[0][1]}-{regions[0][2]}.npz')

data = np.load(path1)

edge_mean = np.mean(data['signal'].T[None,:, :]-data['predictions'], axis=0)
plt.figure(figsize=(8,6), dpi=300)
plt.imshow(edge_mean/edge_mean[:,data['energy_axis']>194].max(axis=1)[:,None], aspect='auto', extent = [data['energy_axis'][0], data['energy_axis'][-1],
                                               0, 1000*data['spatial_axis_y'][-1],], cmap = 'inferno')
plt.xlim(180,230)
plt.axvline(189, 0,1, color = 'gray', linestyle = '--')
plt.axvline(192, 0,1, color = 'gray', linestyle = '--')
plt.text(180.5 + 0, data['spatial_axis_y'][-1]*0.9, r'$p_z(\pi^*)$',
         va='bottom', ha='left', color='white', fontsize=20)
plt.ylabel('Spatial $x$ (nm)')
plt.xlabel('Energy Loss (eV)')
plt.colorbar(label='Normalized Intensity (-)', ticks = [])
plt.xticks([180,230])
plt.yticks([0, 167])
plt.savefig(r'..\figures\ch5_XX_hBN_linescan.pdf',
                    bbox_inches='tight', pad_inches=0)